In [1]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from scipy.stats import norm
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import brier_score_loss, log_loss, accuracy_score

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

In [2]:
#split data: 70/15/15
df = pd.read_parquet("taxi_trips.parquet")
feats = [c for c in df.columns if c != "trip_duration"]
X = df[feats].to_numpy(np.float32)
y = df["trip_duration"].to_numpy(np.float32)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, random_state=0)
X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=0)
print(len(y_tr), len(y_va), len(y_te))

366487 78533 78534


In [3]:
#baselines
baselines = {
    "mean": DummyRegressor(strategy="mean"),
    "linear regression": LinearRegression(),
    "gradient boosting": HistGradientBoostingRegressor(random_state=0),
}
base_pred, base_sigma = {}, {}
for name, m in baselines.items():
    m.fit(X_tr, y_tr)
    base_pred[name] = m.predict(X_te)
    base_sigma[name] = np.std(y_tr - m.predict(X_tr))

def gauss_nll(y, mu, sigma):
    return np.mean((y - mu) ** 2 / (2 * sigma ** 2) + np.log(sigma) + 0.5 * np.log(2 * np.pi))

In [4]:
#(a) link: mu = g1, sigma = softplus(g2)
class GaussianMLP(nn.Module):
    def __init__(self, in_dim, hidden_dims, eps=1e-6):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers.append(nn.Linear(prev, 2))
        self.g = nn.Sequential(*layers)
        self.eps = eps

    def forward(self, x):
        out = self.g(x)
        return out[:, 0], F.softplus(out[:, 1]) + self.eps

#(b) loss: Gaussian NLL
def gaussian_nll(y, mu, sigma):
    return ((y - mu) ** 2 / (2 * sigma ** 2) + torch.log(sigma)).mean()

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        total += gaussian_nll(yb, *model(xb)).item() * len(xb)
    return total / len(loader.dataset)

In [5]:
#(c) train with early stopping on validation NLL
y_mean, y_std = y_tr.mean(), y_tr.std()
t = torch.tensor
train_loader = DataLoader(TensorDataset(t(X_tr), t((y_tr - y_mean) / y_std)), batch_size=512, shuffle=True)
val_loader = DataLoader(TensorDataset(t(X_va), t((y_va - y_mean) / y_std)), batch_size=4096)

model = GaussianMLP(X.shape[1], [128, 128, 64]).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
best, best_state, bad, patience = float("inf"), None, 0, 8
for epoch in range(60):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        gaussian_nll(yb, *model(xb)).backward()
        opt.step()
    v = evaluate(model, val_loader)
    print(f"epoch {epoch + 1:2d}  val NLL {v:.4f}")
    if v < best:
        best, best_state, bad = v, copy.deepcopy(model.state_dict()), 0
    else:
        bad += 1
        if bad >= patience:
            break
model.load_state_dict(best_state)

epoch  1  val NLL -0.2306


epoch  2  val NLL -0.2072


epoch  3  val NLL -0.2461


epoch  4  val NLL -0.2608


epoch  5  val NLL -0.2554


epoch  6  val NLL -0.2710


... (9 outputs left out)

epoch 16  val NLL -0.2684


epoch 17  val NLL -0.2833


epoch 18  val NLL -0.2714


<All keys matched successfully>

In [6]:
#test results
model.eval()
with torch.no_grad():
    mu, sigma = model(t(X_te).to(device))
mu = mu.cpu().numpy() * y_std + y_mean
sigma = sigma.cpu().numpy() * y_std

rows = {}
for name in baselines:
    r = y_te - base_pred[name]
    rows[name] = [np.sqrt(np.mean(r ** 2)), np.mean(np.abs(r)), gauss_nll(y_te, base_pred[name], base_sigma[name])]
r = y_te - mu
rows["Gaussian MLP"] = [np.sqrt(np.mean(r ** 2)), np.mean(np.abs(r)), gauss_nll(y_te, mu, sigma)]
print(pd.DataFrame(rows, index=["RMSE", "MAE", "NLL"]).T.round(3))

                    RMSE    MAE    NLL
mean               7.688  5.664  3.459
linear regression  7.473  5.554  3.430
gradient boosting  4.285  2.982  2.874
Gaussian MLP       3.898  2.454  2.621


In [7]:
#(d) P(Y < 45) = Phi((45 - mu) / sigma)
thr = 45
p_mlp = norm.cdf((thr - mu) / sigma)
p_base = {n: norm.cdf((thr - base_pred[n]) / base_sigma[n]) for n in baselines}

In [8]:
#(e) Brier, log-loss, accuracy at 0.5
y_bin = (y_te < thr).astype(int)
eps = 1e-12
scores = {}
for name, p in {**p_base, "Gaussian MLP": p_mlp}.items():
    p = np.clip(p, eps, 1 - eps)
    scores[name] = [brier_score_loss(y_bin, p), log_loss(y_bin, p), accuracy_score(y_bin, p > 0.5)]
print("fraction of test trips < 45 min:", y_bin.mean().round(4))
print(pd.DataFrame(scores, index=["Brier", "log-loss", "accuracy"]).T.round(4))

fraction of test trips < 45 min: 0.9957
                    Brier  log-loss  accuracy
mean               0.0043    0.0532    0.9957
linear regression  0.0043    0.0445    0.9957
gradient boosting  0.0031    0.0318    0.9967
Gaussian MLP       0.0030    0.0213    0.9965
